In [ ]:
import sympy as sp, numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display, clear_output, HTML

display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def symbolic_dtft_from_scratch():
    n, N, r, omega = sp.symbols('n N r omega', real=True, integer=True)
    alpha = sp.symbols('alpha', real=True)

    r_expr = alpha * sp.exp(-sp.I * omega)
    geometric_sum = (1 - r**(N + 1)) / (1 - r)
    finite_weighted = sp.factor(sp.simplify(r * sp.diff(geometric_sum, r)))
    weighted_limit = sp.simplify(r / (1 - r)**2)
    
    X_omega = weighted_limit.subs(r, r_expr)
    X_omega_simplified = sp.factor(sp.simplify(X_omega))
    X_real, X_imag = sp.simplify(sp.re(X_omega_simplified)), sp.simplify(sp.im(X_omega_simplified))
    magnitude_sym = sp.simplify(sp.sqrt(X_real**2 + X_imag**2))

    print("--- Symbolically Derived Results ---")
    display(HTML(r"<b>Finite symbolic sum =</b> $" + sp.latex(finite_weighted) + "$"))
    display(HTML(r"<b>Infinite weighted geometric sum =</b> $" + sp.latex(weighted_limit) + "$"))
    display(HTML(r"<b>DTFT $X(e^{j\omega}) =$</b> $" + sp.latex(X_omega_simplified) + "$"))
    display(HTML(r"<b>Real part =</b> $" + sp.latex(X_real) + "$"))
    display(HTML(r"<b>Imaginary part =</b> $" + sp.latex(X_imag) + "$"))
    display(HTML(r"<b>Magnitude $|X(e^{j\omega})| =$</b> $" + sp.latex(magnitude_sym) + "$"))

    f_real, f_imag, f_mag = sp.lambdify((omega, alpha), X_real, modules='numpy'), sp.lambdify((omega, alpha), X_imag, modules='numpy'), sp.lambdify((omega, alpha), magnitude_sym, modules='numpy')

    def update_plots(alpha_val):
        clear_output(wait=True)
        omega_vals = np.linspace(-3 * np.pi, 3 * np.pi, 4000)
        real_vals, imag_vals, mag_vals = np.asarray(f_real(omega_vals, alpha_val), dtype=float), np.asarray(f_imag(omega_vals, alpha_val), dtype=float), np.asarray(f_mag(omega_vals, alpha_val), dtype=float)
        phase_vals = np.unwrap(np.angle(real_vals + 1j * imag_vals))

        fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        # Magnitude
        axes[0].plot(omega_vals / np.pi, mag_vals, color='red', lw=2.5, label=f'Magnitude, alpha = {alpha_val:.2f}')
        axes[0].set_title(f'DTFT Magnitude Spectrum of $x[n] = n\\alpha^n u[n]$    ($\\alpha = {alpha_val:.2f}$)', fontsize=11, fontweight='bold')
        axes[0].set_ylabel('Magnitude', fontsize=10)
        axes[0].set_xlim(-3, 3)
        axes[0].set_xticks([-3, -2, -1, 0, 1, 2, 3])
        axes[0].set_xticklabels(['-3π', '-2π', '-π', '0', 'π', '2π', '3π'])
        axes[0].grid(True, linestyle='--', alpha=0.6)
        axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)

        # Phase
        axes[1].plot(omega_vals / np.pi, phase_vals, color='red', lw=2.5, label=f'Phase, alpha = {alpha_val:.2f}')
        axes[1].set_title('DTFT Phase Spectrum', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Normalized Frequency  omega / pi', fontsize=10)
        axes[1].set_ylabel('Phase (radians)', fontsize=10)
        axes[1].set_xlim(-3, 3)
        axes[1].set_xticks([-3, -2, -1, 0, 1, 2, 3])
        axes[1].set_xticklabels(['-3π', '-2π', '-π', '0', 'π', '2π', '3π'])
        axes[1].grid(True, linestyle='--', alpha=0.6)
        axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)

        plt.subplots_adjust(hspace=0.3, right=0.85)
        plt.show()

    alpha_slider = widgets.FloatSlider(value=0.5, min=-0.95, max=0.95, step=0.05, description='Parameter alpha:', style={'description_width': 'initial'})
    ui = widgets.VBox([alpha_slider])
    out = widgets.interactive_output(update_plots, {'alpha_val': alpha_slider})
    
    # Εμφάνιση του UI μία φορά εδώ
    display(ui, out)

symbolic_dtft_from_scratch()